In [ ]:
import mlflow
from pathlib import Path
from mlflow.tracking import MlflowClient


#the only model that is registered in the model registry is the random forest model, so we can load it directly from the model registry->model registry->is released
 #which base
mlflow.set_tracking_uri("sqlite:///mlflow.db")
#which run 
from mlflow.tracking import MlflowClient

run_id = "25bd827ce16d409480c09b0dd37626ea"
client = MlflowClient()
# 3. the model
model = mlflow.sklearn.load_model("models:/wine_quality_rf/1")

# 4. 3JSON
feature_roles = mlflow.artifacts.load_dict(f"runs:/{run_id}/feature_roles.json")
feature_order = mlflow.artifacts.load_dict(f"runs:/{run_id}/canonical_feature_order.json")["canonical_feature_order"]
split = mlflow.artifacts.load_dict(f"runs:/{run_id}/split_indices.json")

print(feature_roles)
print(feature_order)
print(len(split["train_idx"]), len(split["test_idx"]))

phase 5 
1.load the model and data
2.reconstruct the exact training split 
3.layer 1 - per-feauture percentile bounds
4.layer 2 -mahalanobis validity region (ledoitwolf)

In [ ]:
#loading the x train 
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() #to make it accesible from both notebooks and scripts
print("PROJECT_ROOT:", PROJECT_ROOT)
import pandas as pd
wine = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "winequality-white (1).csv", sep=";")

wine.head()

wine=wine.drop_duplicates().reset_index(drop=True)
X_full=wine[feature_order] #we can use the feature order to select the features in the same order as they were used in the model training
X_train=X_full.loc[split["train_idx"]] #we use split indices to select the training data




In [ ]:
print(X_train.shape)



In [ ]:
#Percentile bounds -we dont use min max because they are sensitive to outliers, we use the 1st and 99th percentile instead

lo=X_train.quantile(0.01)
hi=X_train.quantile(0.99)
#for each feature we create a dictionary with the lower and upper bounds
bounds={f: [float(lo[f]),float(hi[f])] for f in feature_order}
for f in feature_order:
    print(f"{f:24s} [{bounds[f][0]:8.3f},{bounds[f][1]:8.3f}]")



we see in the residual sugar a big gap the 0.9 is dry but the 18.75 is sweet 


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(X_train["residual sugar"], bins=60, color="#8B6B8B", edgecolor="white")
axes[0].axvline(bounds["residual sugar"][0], color="crimson", ls="--", label="1st pct")
axes[0].axvline(bounds["residual sugar"][1], color="crimson", ls="--", label="99th pct")
axes[0].set_xlabel("residual sugar (g/L)")
axes[0].set_ylabel("count")
axes[0].set_title("Right-skewed, not Gaussian")
axes[0].legend()

axes[1].hist(X_train["alcohol"], bins=60, color="#C9A227", edgecolor="white")
axes[1].set_xlabel("alcohol (% vol)")
axes[1].set_title("Compare: closer to symmetric")

plt.tight_layout()
plt.show()


we see almost no residual sugar because it finishes and also we see quantization (alcohol that does not be measured the whole time )

In [ ]:
import matplotlib.pyplot as plt
from scipy import stats

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.ravel()

for ax, feat in zip(axes, feature_order):
    x = X_train[feat]
    ax.hist(x, bins=50, color="#8B6B8B", edgecolor="white", linewidth=0.3)
    ax.axvline(bounds[feat][0], color="crimson", ls="--", lw=1)
    ax.axvline(bounds[feat][1], color="crimson", ls="--", lw=1)
    ax.set_title(f"{feat}\nskew = {stats.skew(x):.2f}", fontsize=10)
    ax.tick_params(labelsize=8)

axes[-1].axis("off")   

fig.suptitle("Phase 5 — training-set marginals with 1st/99th percentile bounds", y=1.00)
plt.tight_layout()
plt.savefig("../results/figures/phase5_all_marginals.png", dpi=150, bbox_inches="tight")
plt.show()

the chlorides are signatures of how close they are to the sea so thats why put them as context 

In [ ]:
import numpy as np
#we use ledoit wolf to estimate the covariance matrix

from sklearn.covariance import LedoitWolf

lw=LedoitWolf().fit(X_train)#we just fit the model to the training data, we dont need to transform the data because we are only interested in the covariance matrix

#finding the 
cond_raw=np.linalg.cond(np.cov(X_train.T,ddof=1))#ddof =1 because we want the sample covariance matrix, not the population covariance matrix,we want also transposed
cond_lw=np.linalg.cond(lw.covariance_)

print(f"shrinkage δ:      {lw.shrinkage_:.4f}")
print(f"cond raw:         {cond_raw:.3e}")
print(f"cond LedoitWolf:  {cond_lw:.3e}")
print(f"improvement:      {cond_raw/cond_lw:.1f}×")

assert cond_lw < cond_raw, "shrinkage did not improve conditioning"


free SO₂ causally influences volatile acidity through microbial spoilage control (Tian et al., 2026), so proposed reformulations that reduce SO₂ while holding VA fixed are not physically realisable

Tial et al used grape and oak tannins to replaced SO2 in white and red wine -> Tannin addition reduced the SO2 requirement, ensured smooth fermentation,and lowered the volatile acidity.


In [ ]:
legal = {
    "total sulfur dioxide": {
        "cap_low_sugar":  200.0,   # residual sugar < 5 g/L
        "cap_high_sugar": 250.0,   # residual sugar >= 5 g/L
        "sugar_threshold_gL": 5.0,
        "unit": "mg/L",
        "source": "Reg (EU) 2019/934, Annex I, Part B",
    },
    "volatile acidity": {
        "cap": 1.08,
        "unit": "g/L acetic acid (18 meq/L)",
        "source": "Reg (EU) 2019/934, Annex I, Part C",
    },
}


In [ ]:
mlflow.end_run()
with mlflow.start_run(run_id="25bd827ce16d409480c09b0dd37626ea"):
    mlflow.log_dict(legal, "constraints/legal.json")

In [ ]:
#lets see the mahalanobis distance of the training data 
#the mahalanobis distance measures how many standard deviations away a point is from the mean of a distribution, taking into account the correlations between the variables. It is useful for identifying outliers in multivariate data.


#we use them in ml because if a point has a high mahalanobis distance its likely an outlier 

d2_train_cube=lw.mahalanobis(X_train)

d2_train_cube_df=pd.DataFrame( lw.mahalanobis(X_train), columns=["mahalanobis_distance"], index=X_train.index)

d2_train_cube_df.describe(percentiles=[0.01,0.05,0.25,0.5,0.75,0.95,0.99])


d2_train_cube_df.head(10)

In [ ]:
print(f"n: {len(d2_train_cube_df)}")

print(f"median: {np.median(d2_train_cube_df['mahalanobis_distance']):.2f}")
print(f"p95:    {np.quantile(d2_train_cube_df['mahalanobis_distance'], 0.95):.2f}")
print(f"max:    {d2_train_cube_df['mahalanobis_distance'].max():.2f}")

plt.hist(d2_train_cube_df["mahalanobis_distance"], bins=80)
plt.axvline(np.quantile(d2_train_cube_df["mahalanobis_distance"], 0.95), color="k", ls="--")
plt.xlabel("squared Mahalanobis distance"); plt.show()



In [ ]:
#the biggest outlier 

X_train.loc[d2_train_cube_df["mahalanobis_distance"].idxmax()]

The highest Mahalanobis distance in the training set (d² = 246.76, vs. threshold 10.15) belongs to a wine with free SO₂ = 289 mg/L and total SO₂ = 440 mg/L, the dataset maxima for both features simultaneously. Residual sugar is 2.9 g/L, so the applicable EU cap is 200 mg/L, the wine exceeds it by more than a factor of two.Maybe these sample as not ready for the market maybe they are from lab testing

In [ ]:
#lets save them all 
mlflow.end_run()

with mlflow.start_run(run_id="25bd827ce16d409480c09b0dd37626ea"):
     d2 = d2_train_cube_df["mahalanobis_distance"]
     mahal = {
    "location_": lw.location_.tolist(),
    "precision_": lw.precision_.tolist(),
    "threshold_d2": float(np.quantile(d2, 0.95)),
    "threshold_rule": "empirical 95th percentile of training d2",
    "shrinkage": float(lw.shrinkage_),
    "feature_order": feature_order,
    "n_train": int(len(d2)),
    }

   



In [ ]:
#lets log all of them 
mlflow.end_run()
run_id = "25bd827ce16d409480c09b0dd37626ea"
with mlflow.start_run(run_id="25bd827ce16d409480c09b0dd37626ea"):
    mlflow.log_dict(
        {"bounds": bounds, "rule": "1st–99th percentile of training set",
         "feature_order": feature_order, "n_train": int(len(X_train))},
        "constraints/bounds.json")
   
    mlflow.log_dict(mahal, "constraints/mahal.json")
print([f.path for f in MlflowClient().list_artifacts(run_id, "constraints")])

In [ ]:
#we will learn from the data the rule that connects the density with the consistency\
#we ask what density it can have


import statsmodels.api as sm

y_dens=X_train["density"]
X_dens=sm.add_constant(X_train[["alcohol", "residual sugar"]]) 
#if we dont add constant (a column of ones) the model will not have an intercept, and the regression line will be forced to go through the origin, which is not appropriate for this data. We want to allow the model to learn an intercept term that captures the baseline density when alcohol and residual sugar are zero.
#the origin will be like wine with 0% alchol 0 sugar and density 0

#density is density=bo(water)-b1*alcohol(make it less dense)-b2*sugar(make it more dense)

ols=sm.OLS(y_dens,X_dens).fit()

print(ols.summary())
print(ols.params)
print(f"\nR²    = {ols.rsquared:.4f}")
print(f"sigma = {ols.resid.std(ddof=3):.6f} g/cm³")



91% of the density variation is explained by just two variables. It confirms that density is not free , it is almost a function of the other two.
Band width. ±3σ spans 0.00255 g/cm³, against a total density range of 0.011 in the training set. The physically admissible window is therefore under half the width of the Layer 1 box bounds, a large volume of the search space is legal per-feature but chemically impossible.

Residual diagnostics. Residuals are right-skewed (skew = 0.927, kurtosis = 4.72), so the Gaussian ±3σ ≈ 99.7% rule does not hold

In [ ]:
sigma = float(ols.resid.std(ddof=3))
z = (ols.resid / sigma).abs()

print(f"outside ±3σ: {(z > 3).sum()} / {len(z)}  ({(z>3).mean()*100:.2f}%)")

i = d2_train_cube_df["mahalanobis_distance"].idxmax()
print(f"wine {i}: |z| = {z.loc[i]:.2f}   (d² = {d2_train_cube_df.loc[i, 'mahalanobis_distance']:.1f})")

ols.resid ->real density -predicted denstity
sigma ->typical diaspora of these residuals
also d^2 tells us how 'how far, in s, in 11 directions at once'
the z of 0.34 tells us that this wine is one-third of a standard deviation away from the norm 



In [ ]:
#the density band 
#the the permissible value window of the density, given the alcohol and the residual sugar.

density_band = {
    "target": "density",
    "predictors": ["alcohol", "residual sugar"],
    "coef": {
        "const": float(ols.params["const"]),
        "alcohol": float(ols.params["alcohol"]),
        "residual sugar": float(ols.params["residual sugar"]),
    },
    "sigma": sigma,
    "k": 3.0,
    "rule": "|density - (const + b_alcohol*alcohol + b_sugar*residual_sugar)| <= k*sigma",
    "r2": float(ols.rsquared),
    "empirical_exceedance": float((z > 3).mean()),
    "resid_skew": float(stats.skew(ols.resid)),
    "n_train": int(len(y_dens)),
    "rationale": "restores the density-composition law relaxed by LedoitWolf shrinkage",
}

client.log_dict(run_id, density_band, "constraints/density_band.json")
print([f.path for f in client.list_artifacts(run_id, "constraints")])

In [ ]:
from src.modeling.constraints import check,load_artifacts
run_id = "25bd827ce16d409480c09b0dd37626ea"
